# 4. Context Parallelism

Context parallelism splits a long sequence across GPUs along the
**token dimension**.

**Core idea**
- GPU 0 owns one slice of the sequence
- GPU 1 owns another slice
- each GPU produces outputs for its own tokens
- attention still needs information from the full context

This notebook shows the basic idea with a tiny single-head attention
block.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Why this is more advanced

With context parallelism, the sequence is split, but each token may
still need to look at other tokens.

That means:
- some operations are local and easy
- attention usually needs communication so every shard can use the
  full set of keys and values

We simulate that communication by gathering keys and values from all
shards before each local attention computation.

In [ ]:
import math


class TinyAttention(nn.Module):
    """Single-head attention kept intentionally small and readable."""

    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def project(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.q_proj(x), self.k_proj(x), self.v_proj(x)

    def attend(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
    ) -> torch.Tensor:
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.hidden_size)
        weights = scores.softmax(dim=-1)
        return weights @ v

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        q, k, v = self.project(x)
        context = self.attend(q, k, v)
        return self.out_proj(context)


hidden_size = 4
sequence_length = 6
num_shards = 2

attention = TinyAttention(hidden_size)
x = torch.randn(1, sequence_length, hidden_size)

full_output = attention(x)

print("Input shape:", tuple(x.shape))
print("Full attention output shape:", tuple(full_output.shape))

In [ ]:
x_shards = x.chunk(num_shards, dim=1)
local_qkv = [attention.project(shard) for shard in x_shards]

all_k = torch.cat([k for _, k, _ in local_qkv], dim=1)
all_v = torch.cat([v for _, _, v in local_qkv], dim=1)

local_outputs = []
for shard_id, (q_local, _, _) in enumerate(local_qkv):
    local_context = attention.attend(q_local, all_k, all_v)
    local_output = attention.out_proj(local_context)
    local_outputs.append(local_output)
    print(
        f"Shard {shard_id}: "
        f"local queries={tuple(q_local.shape)}, "
        f"global keys={tuple(all_k.shape)}"
    )

context_parallel_output = torch.cat(local_outputs, dim=1)

print()
print(
    "Max difference after context parallelism:",
    (full_output - context_parallel_output).abs().max().item(),
)
assert torch.allclose(full_output, context_parallel_output, atol=1e-7)
print("Context-parallel attention matches the full attention output.")

## Takeaways

- Context parallelism helps when the **sequence length** is the main
  memory problem.
- Each shard owns only part of the tokens.
- Attention often requires communication because local queries may
  need global keys and values.
- This idea becomes very useful for long-context language models.